<a href="https://colab.research.google.com/github/GeorgeKontsevik/GCN_forecast_urban_services_loads/blob/main/ver5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install osmnx torch_geometric pygeoops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.3/83.3 kB 9.2 MB/s eta 0:00:00


# imports

In [2]:
import random

In [ ]:
import numpy as np
import networkx as nx
import osmnx as ox
import pickle
import h5py
import torch
from shapely.geometry import LineString
from torch.utils.data import Dataset, DataLoader
from models import DynamicModel
from config import possible_models
from data_loader import collate_data
from dataset import IndexedDataset
from torch.utils.data import Dataset, DataLoader

In [ ]:
import numpy as np
import networkx as nx
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon
from shapely.plotting import plot_polygon, plot_line
import torch
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch

In [ ]:
# !pip install rtree

In [ ]:
import osmnx as ox
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import split
import networkx as nx
import cv2
import numpy as np
import pygeoops
from tqdm import tqdm
from rtree import index
from shapely.affinity import translate
import math

# grid

In [ ]:
def split_graph_by_grid(G, grid_step=1000):

    # Получаем границы графа через преобразование в GeoDataFrame
    nodes, edges = ox.graph_to_gdfs(G)
    bounds = nodes.total_bounds  # [minx, miny, maxx, maxy]

    minx, miny, maxx, maxy = bounds

    # Создаем сетку
    x_coords = np.arange(minx, maxx, grid_step)
    y_coords = np.arange(miny, maxy, grid_step)

    # Добавляем последнюю координату чтобы покрыть всю область
    if len(x_coords) == 0 or x_coords[-1] < maxx:
        x_coords = np.append(x_coords, maxx)
    if len(y_coords) == 0 or y_coords[-1] < maxy:
        y_coords = np.append(y_coords, maxy)

    # Словарь для хранения подграфов и полигонов
    subgraphs = {}
    idx = index.Index()
    cell_bounds = {}

    # Создаем ячейки сетки и подграфы
    for i in range(len(x_coords) - 1):
        for j in range(len(y_coords) - 1):
            cell_polygon = Polygon([
                (x_coords[i], y_coords[j]),
                (x_coords[i+1], y_coords[j]),
                (x_coords[i+1], y_coords[j+1]),
                (x_coords[i], y_coords[j+1])
            ])

            # Создаем новый подграф для этой ячейки
            subgraph = nx.MultiDiGraph()
            subgraph.graph['crs'] = G.graph['crs']

            subgraphs[(i, j)] = {
                'graph': subgraph,
                'polygon': cell_polygon
            }

    for i, (key, cell_data) in enumerate(subgraphs.items()):
        bounds = cell_data['polygon'].bounds  # (min_x, min_y, max_x, max_y)
        idx.insert(i, bounds)
        cell_bounds[i] = (key, cell_data)

    # Словарь для отслеживания, в какой ячейке находится каждый узел
    node_to_cell = {}

    # Сначала добавляем все узлы в соответствующие ячейки
    for node, data in tqdm(G.nodes(data=True), desc="Assigning nodes to cells"):
        point = Point(data['x'], data['y'])

        potential_cells = list(idx.intersection((data['x'], data['y'], data['x'], data['y'])))

        # Находим ячейку, в которой находится узел
        assigned_to_cell = False

        for cell_idx in potential_cells:
            key, cell_data = cell_bounds[cell_idx]
            if cell_data['polygon'].contains(point):
                cell_data['graph'].add_node(node, **data)
                node_to_cell[node] = key
                assigned_to_cell = True
                break

        # for key, cell_data in subgraphs.items():
        #     if cell_data['polygon'].contains(point):
        #         # Добавляем узел в подграф
        #         cell_data['graph'].add_node(node, **data)
        #         node_to_cell[node] = key
        #         assigned_to_cell = True
        #         break

        # Если узел не попал в какую-то ячейку (например, на границе),
        # добавляем его в первую подходящую ячейку
        if not assigned_to_cell:
            for cell_idx in potential_cells:
                key, cell_data = cell_bounds[cell_idx]
                if cell_data['polygon'].intersects(point):
                    cell_data['graph'].add_node(node, **data)
                    node_to_cell[node] = key
                    break

            # for key, cell_data in subgraphs.items():
            #     if cell_data['polygon'].intersects(point):
            #         cell_data['graph'].add_node(node, **data)
            #         node_to_cell[node] = key
            #         break

    cell_idx_es = index.Index()
    cell_data_by_idx = {}
    for i, (key, cell_data) in enumerate(subgraphs.items()):
        bounds = cell_data['polygon'].bounds  # (min_x, min_y, max_x, max_y)
        cell_idx_es.insert(i, bounds)
        cell_data_by_idx[i] = (key, cell_data)

    edges_with_geometry = []
    for u, v, key, data in G.edges(keys=True, data=True):
        if 'geometry' in data:
            line = data['geometry']
        else:
            u_x, u_y = G.nodes[u]['x'], G.nodes[u]['y']
            v_x, v_y = G.nodes[v]['x'], G.nodes[v]['y']
            line = LineString([(u_x, u_y), (v_x, v_y)])
        edges_with_geometry.append((u, v, key, data, line))

    for u, v, key, data, line in tqdm(edges_with_geometry, desc="Assigning edges to cells"):
        # Проверяем, в каких ячейках находятся узлы u и v
        u_cell = node_to_cell.get(u)
        v_cell = node_to_cell.get(v)

        # Находим потенциальные ячейки для ребра
        line_bounds = line.bounds  # (minx, miny, maxx, maxy)
        potential_cell_indices = list(cell_idx_es.intersection(line_bounds))

        for cell_idx in potential_cell_indices:
            cell_key, cell_data = cell_data_by_idx[cell_idx]
            cell_polygon = cell_data['polygon']

            cell_bounds = cell_polygon.bounds
            if (line_bounds[2] > cell_bounds[2] or line_bounds[2] < cell_bounds[0] or
                line_bounds[3] < cell_bounds[1] or line_bounds[1] > cell_bounds[3]):
                continue  # Нет пересечения по bounding box

            if not line.intersects(cell_polygon):
                continue  # Нет пересечения с полигоном ячейки

            subgraph = cell_data['graph']
            intersetion = line.intersection(cell_polygon)
            if intersetion.is_empty:
                continue
            if intersetion.geom_type == 'LineString':
                segments = [intersetion]
            elif intersetion.geom_type == 'MultiLineString':
                segments = list(intersetion.geoms)
            else:
                continue

            for segment in segments:
                start_coord = segment.coords[0]
                end_coord = segment.coords[-1]

                start_node_exists = u in subgraph.nodes() if cell_key == u_cell else False
                end_node_exists = v in subgraph.nodes() if cell_key == v_cell else False

                if start_node_exists and end_node_exists:
                    start_node_id = u
                    end_node_id = v
                elif start_node_exists:
                    start_node_id = u
                    end_node_id = f"end_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                    subgraph.add_node(end_node_id, x=end_coord[0], y=end_coord[1])
                elif end_node_exists:
                    start_node_id = f"start_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                    end_node_id = v
                    subgraph.add_node(start_node_id, x=start_coord[0], y=start_coord[1])
                else:
                    start_node_id = f"start_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                    end_node_id = f"end_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                    subgraph.add_node(start_node_id, x=start_coord[0], y=start_coord[1])
                    subgraph.add_node(end_node_id, x=end_coord[0], y=end_coord[1])

                edge_data = data.copy()
                edge_data['geometry'] = segment
                edge_data['original_nodes'] = (u, v)
                edge_data['segment_id'] = f"{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                edge_data['original_edge_key'] = key

                subgraph.add_edge(start_node_id, end_node_id, **edge_data)

    # # Затем обрабатываем рёбра
    # for u, v, key, data in tqdm(G.edges(keys=True, data=True), desc="Assigning edges to cells"):
    #     # Проверяем, в каких ячейках находятся узлы u и v
    #     u_cell = node_to_cell.get(u)
    #     v_cell = node_to_cell.get(v)

    #     # Получаем геометрию ребра
    #     if 'geometry' in data:
    #         line = data['geometry']
    #     else:
    #         # Создаем линию из координат узлов
    #         u_x, u_y = G.nodes[u]['x'], G.nodes[u]['y']
    #         v_x, v_y = G.nodes[v]['x'], G.nodes[v]['y']
    #         line = LineString([(u_x, u_y), (v_x, v_y)])

    #     # Находим все ячейки, через которые проходит ребро
    #     intersecting_cells = []
    #     for cell_key, cell_data in subgraphs.items():
    #         if line.intersects(cell_data['polygon']):
    #             intersecting_cells.append((cell_key, cell_data))

    #     # Для каждой пересекающей ячейки добавляем сегмент ребра
    #     for cell_key, cell_data in intersecting_cells:
    #         subgraph = cell_data['graph']
    #         cell_polygon = cell_data['polygon']

    #         # Вычисляем пересечение линии с ячейкой
    #         intersection = line.intersection(cell_polygon)

    #         if intersection.is_empty:
    #             continue

    #         if intersection.geom_type == 'LineString':
    #             segments = [intersection]
    #         elif intersection.geom_type == 'MultiLineString':
    #             segments = list(intersection.geoms)
    #         else:
    #             continue

    #         for segment in segments:
    #             # Проверяем, нужно ли создавать новые узлы для сегмента
    #             # Если оба конца сегмента находятся в ячейке, используем оригинальные узлы
    #             # Если только один конец - создаём новый узел для другого конца
    #             # Если ни один - создаём оба новых узла

    #             start_coord = segment.coords[0]
    #             end_coord = segment.coords[-1]

    #             # Проверяем, находятся ли оригинальные узлы в этой ячейке
    #             start_node_exists = u in subgraph.nodes() if cell_key == u_cell else False
    #             end_node_exists = v in subgraph.nodes() if cell_key == v_cell else False

    #             # Определяем ID узлов для этого сегмента
    #             if start_node_exists and end_node_exists:
    #                 # Оба узла уже существуют в этом подграфе
    #                 start_node_id = u
    #                 end_node_id = v
    #             elif start_node_exists:
    #                 # Только начальный узел существует, создаём конечный
    #                 start_node_id = u
    #                 end_node_id = f"end_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
    #                 subgraph.add_node(end_node_id, x=end_coord[0], y=end_coord[1])
    #             elif end_node_exists:
    #                 # Только конечный узел существует, создаём начальный
    #                 start_node_id = f"start_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
    #                 end_node_id = v
    #                 subgraph.add_node(start_node_id, x=start_coord[0], y=start_coord[1])
    #             else:
    #                 # Ни один узел не существует в этой ячейке, создаём оба
    #                 start_node_id = f"start_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
    #                 end_node_id = f"end_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
    #                 subgraph.add_node(start_node_id, x=start_coord[0], y=start_coord[1])
    #                 subgraph.add_node(end_node_id, x=end_coord[0], y=end_coord[1])

    #             # Создаем копию данных ребра с обновленной геометрией
    #             edge_data = data.copy()
    #             edge_data['geometry'] = segment
    #             edge_data['original_nodes'] = (u, v)
    #             edge_data['segment_id'] = f"{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
    #             edge_data['original_edge_key'] = key

    #             # Добавляем ребро в подграф
    #             subgraph.add_edge(start_node_id, end_node_id, **edge_data)

    # Возвращаем только непустые подграфы с их полигонами
    result = {}
    for key, cell_data in subgraphs.items():
        if len(cell_data['graph'].nodes) > 0:
            result[key] = {
                'graph': cell_data['graph'],
                'polygon': cell_data['polygon']
            }

    return result

In [ ]:
def split_graph_by_grid(G, grid_step=1000):

    # Получаем границы графа через преобразование в GeoDataFrame
    nodes, edges = ox.graph_to_gdfs(G)
    bounds = nodes.total_bounds  # [minx, miny, maxx, maxy]

    minx, miny, maxx, maxy = bounds

    # Создаем сетку
    x_coords = np.arange(minx, maxx, grid_step)
    y_coords = np.arange(miny, maxy, grid_step)

    # Добавляем последнюю координату чтобы покрыть всю область
    if len(x_coords) == 0 or x_coords[-1] < maxx:
        x_coords = np.append(x_coords, maxx)
    if len(y_coords) == 0 or y_coords[-1] < maxy:
        y_coords = np.append(y_coords, maxy)

    # Словарь для хранения подграфов и полигонов
    subgraphs = {}

    # Создаем R-tree индекс для ячеек (узлов)
    node_cell_idx = index.Index()
    node_cell_bounds = {}

    # Создаем ячейки сетки и подграфы
    for i in range(len(x_coords) - 1):
        for j in range(len(y_coords) - 1):
            cell_polygon = Polygon([
                (x_coords[i], y_coords[j]),
                (x_coords[i+1], y_coords[j]),
                (x_coords[i+1], y_coords[j+1]),
                (x_coords[i], y_coords[j+1])
            ])

            # Создаем новый подграф для этой ячейки
            subgraph = nx.MultiDiGraph()
            subgraph.graph['crs'] = G.graph['crs']

            subgraphs[(i, j)] = {
                'graph': subgraph,
                'polygon': cell_polygon
            }

    # Заполняем R-tree индекс для узлов
    for i, (key, cell_data) in enumerate(subgraphs.items()):
        bounds = cell_data['polygon'].bounds  # (min_x, min_y, max_x, max_y)
        node_cell_idx.insert(i, bounds)
        node_cell_bounds[i] = (key, cell_data)

    # Словарь для отслеживания, в какой ячейке находится каждый узел
    node_to_cell = {}

    # Сначала добавляем все узлы в соответствующие ячейки
    for node, data in tqdm(G.nodes(data=True), desc="Assigning nodes to cells"):
        point = Point(data['x'], data['y'])

        potential_cells = list(node_cell_idx.intersection((data['x'], data['y'], data['x'], data['y'])))

        # Находим ячейку, в которой находится узел
        assigned_to_cell = False

        for cell_idx in potential_cells:
            key, cell_data = node_cell_bounds[cell_idx]
            if cell_data['polygon'].contains(point):
                cell_data['graph'].add_node(node, **data)
                node_to_cell[node] = key
                assigned_to_cell = True
                break

        # Если узел не попал в какую-то ячейку (например, на границе),
        # добавляем его в первую подходящую ячейку
        if not assigned_to_cell:
            for cell_idx in potential_cells:
                key, cell_data = node_cell_bounds[cell_idx]
                if cell_data['polygon'].intersects(point):
                    cell_data['graph'].add_node(node, **data)
                    node_to_cell[node] = key
                    break

    # Создаем отдельный R-tree индекс для обработки рёбер
    edge_cell_idx = index.Index()
    edge_cell_data_by_idx = {}

    for i, (key, cell_data) in enumerate(subgraphs.items()):
        bounds = cell_data['polygon'].bounds
        edge_cell_idx.insert(i, bounds)
        edge_cell_data_by_idx[i] = (key, cell_data)

    # Предварительно вычисляем геометрии рёбер
    edges_with_geometry = []
    for u, v, key, data in G.edges(keys=True, data=True):
        if 'geometry' in data:
            line = data['geometry']
        else:
            u_x, u_y = G.nodes[u]['x'], G.nodes[u]['y']
            v_x, v_y = G.nodes[v]['x'], G.nodes[v]['y']
            line = LineString([(u_x, u_y), (v_x, v_y)])
        edges_with_geometry.append((u, v, key, data, line))

    # Обрабатываем рёбра с использованием пространственного индекса
    for u, v, key, data, line in tqdm(edges_with_geometry, desc="Assigning edges to cells"):
        u_cell = node_to_cell.get(u)
        v_cell = node_to_cell.get(v)

        # Находим потенциальные ячейки для ребра
        line_bounds = line.bounds  # (minx, miny, maxx, maxy)
        potential_cell_indices = list(edge_cell_idx.intersection(line_bounds))

        for cell_index in potential_cell_indices:  # переименовал переменную чтобы избежать конфликта
            cell_key, cell_data = edge_cell_data_by_idx[cell_index]
            cell_polygon = cell_data['polygon']

            # Быстрая проверка по bounding box
            cell_bounds = cell_polygon.bounds
            if (line_bounds[0] > cell_bounds[2] or line_bounds[2] < cell_bounds[0] or
                line_bounds[1] > cell_bounds[3] or line_bounds[3] < cell_bounds[1]):
                continue

            if not line.intersects(cell_polygon):
                continue

            subgraph = cell_data['graph']
            intersection = line.intersection(cell_polygon)

            if intersection.is_empty:
                continue

            if intersection.geom_type == 'LineString':
                segments = [intersection]
            elif intersection.geom_type == 'MultiLineString':
                segments = list(intersection.geoms)
            else:
                continue

            for segment in segments:
                start_coord = segment.coords[0]
                end_coord = segment.coords[-1]

                start_node_exists = u in subgraph.nodes() if cell_key == u_cell else False
                end_node_exists = v in subgraph.nodes() if cell_key == v_cell else False

                if start_node_exists and end_node_exists:
                    start_node_id = u
                    end_node_id = v
                elif start_node_exists:
                    start_node_id = u
                    end_node_id = f"end_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                    subgraph.add_node(end_node_id, x=end_coord[0], y=end_coord[1])
                elif end_node_exists:
                    start_node_id = f"start_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                    end_node_id = v
                    subgraph.add_node(start_node_id, x=start_coord[0], y=start_coord[1])
                else:
                    start_node_id = f"start_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                    end_node_id = f"end_{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                    subgraph.add_node(start_node_id, x=start_coord[0], y=start_coord[1])
                    subgraph.add_node(end_node_id, x=end_coord[0], y=end_coord[1])

                edge_data = data.copy()
                edge_data['geometry'] = segment
                edge_data['original_nodes'] = (u, v)
                edge_data['segment_id'] = f"{u}_{v}_{key}_{hash(segment.wkt) % 10000:04d}"
                edge_data['original_edge_key'] = key

                subgraph.add_edge(start_node_id, end_node_id, **edge_data)

    # Возвращаем только непустые подграфы с их полигонами
    result = {}
    for key, cell_data in subgraphs.items():
        if len(cell_data['graph'].nodes) > 0:
            result[key] = {
                'graph': cell_data['graph'],
                'polygon': cell_data['polygon']
            }

    return result

# extract blocks

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, LineString, Point
from shapely.ops import polygonize, linemerge
import geopandas as gpd
import pandas as pd
import numpy as np

def extract_street_blocks(graph_dict):
    graph = graph_dict['graph']
    polygon = graph_dict['polygon']
    nodes, lines = ox.graph_to_gdfs(graph)
    all_lines = lines.union_all()
    lines_buffered = all_lines.buffer(0.0001)
    result_geometry = polygon.difference(lines_buffered)
    # result_geometry = polygon.difference(all_lines)

    if result_geometry.geom_type == 'MultiPolygon':
        polygons = list(result_geometry.geoms)
    else:
        polygons = [result_geometry]

    blocks_gdf = gpd.GeoDataFrame({
        'block_id': range(len(polygons)),
        'geometry': [poly.buffer(0.0001) for poly in polygons]
    }, crs=graph.graph['crs'])

    return blocks_gdf


# create block graph

In [ ]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
from shapely.geometry import Point, LineString, MultiLineString

def create_street_block_graph(gdf_blocks, graph, buffer: float = 0.0001):
    gdf_blocks = gdf_blocks.copy()
    gdf_blocks['centroid'] = gdf_blocks.geometry.centroid

    block_graph = nx.MultiDiGraph()

    block_graph.graph['crs'] = graph.graph['crs']

    for idx, row in gdf_blocks.iterrows():
        block_graph.add_node(
            row['block_id'],
            geometry=row['geometry'],
            centroid=row['centroid'],
            x=row['centroid'].x,
            y=row['centroid'].y
        )

    spatial_index = gdf_blocks.sindex
    gdf_blocks_buffered = gdf_blocks.copy()
    gdf_blocks_buffered['geometry_buffered'] = gdf_blocks_buffered.geometry.buffer(buffer)

    for i, block_i in gdf_blocks.iterrows():
        possible_matches_index = list(spatial_index.intersection(block_i.geometry.bounds))
        possible_matches = gdf_blocks.iloc[possible_matches_index]

        for j, block_j in possible_matches.iterrows():
            if i >= j:
                continue

            geom_i_buffered = gdf_blocks_buffered.loc[i, 'geometry_buffered']
            geom_j_buffered = gdf_blocks_buffered.loc[j, 'geometry_buffered']

            if geom_i_buffered.intersects(geom_j_buffered):
                line_geom = LineString([block_i['centroid'], block_j['centroid']])

                block_graph.add_edge(
                    block_i['block_id'],
                    block_j['block_id'],
                    geometry=line_geom,
                    length=line_geom.length
                )

    return block_graph, gdf_blocks



# features

In [ ]:
'''
area — площадь (масштабируется делением на 10000)

circuity — извилистость

concavity — вогнутость

number_of_linestrings — количество линий

formfacter — фактор формы

rectanglarity — прямоугольность

elongation — удлинение

degree — степень узла
'''

feature_keys = ['+area', 'circuity', 'concavity', 'number_of_linestrings', '+formfacter', 'rectanglarity',
                        '+elongation', '+degree']

In [ ]:
def compute_features(block_graph, gdf_blocks, graph):

    def mec(vertices):
        points = np.array(vertices, dtype=np.float32)
        return cv2.minEnclosingCircle(points)

    nodes_street_graph, lines_street_graph = ox.graph_to_gdfs(graph)
    list_geom = list(nodes_street_graph.geometry)
    # points_series = gpd.GeoSeries(list_geom)
    # lines_series = gpd.GeoSeries(list(lines_street_graph.geometry))

    for node in block_graph.nodes:
        block_graph.nodes[node]['number_of_linestrings'] = 0
        polygon = block_graph.nodes[node]['geometry'] # к какому полигону (кварталу) относится этот узел
        area = polygon.area
        length = polygon.length

        if polygon.geom_type == 'MultiPolygon':
            all_points = []
            for polygon in polygon.geoms:
                all_points.extend(list(polygon.exterior.coords))
                for interior in polygon.interiors:
                    all_points.extend(list(interior.coords))

            points = np.array(all_points, dtype=np.float32)
            circle_coords = mec(points)
        else:
            circle_coords = mec(list(polygon.exterior.coords))

        points_coords = []
        lines_edges = []
        for node_street_graph in graph.nodes:
            point = Point(graph.nodes[node_street_graph]['x'], graph.nodes[node_street_graph]['y'])
            if polygon.intersects(point) and graph.degree[node_street_graph] == 1:
                block_graph.nodes[node]['number_of_linestrings'] += 1
            else:
                points_coords += [node_street_graph]

        for i1, p1 in enumerate(points_coords):
            for i2, p2 in enumerate(points_coords[i1+1:]):
                if graph.has_edge(p1, p2, 0):
                    g = graph.edges[p1, p2, 0]['geometry']
                    lines_edges.append(g)

        len_edges = len(lines_edges)

        circuity = 0
        for line in lines_edges:
            length = line.length
            line_coo = line.coords
            s_p = Point(line_coo[0])
            e_p = Point(line_coo[-1])
            straight_line_length = LineString([s_p, e_p]).length
            circuity += (length / straight_line_length) / len_edges if len_edges > 0 and straight_line_length > 0 else 0

        # print("circuity", circuity)


        circle = Point(circle_coords[0]).buffer(circle_coords[1])
        b_box_minx, b_box_miny, b_box_maxx, b_box_maxy = polygon.bounds
        b_box_width = b_box_maxx - b_box_minx
        b_box_height = b_box_maxy - b_box_miny
        convex_hull = polygon.convex_hull
        convex_hull_area = convex_hull.area
        min_rot_rec = polygon.minimum_rotated_rectangle
        min_rot_rec_area = min_rot_rec.area

        block_graph.nodes[node]['area'] = area / 10000
        block_graph.nodes[node]['circuity'] = circuity
        block_graph.nodes[node]['formfacter'] = area / circle.area if circle.area > 0 else 0
        # print("formfacter", block_graph.nodes[node]['formfacter'])
        # print("circle area", circle.area, circle)
        block_graph.nodes[node]['elongation'] = b_box_height / b_box_width if b_box_width > 0 else 0
        block_graph.nodes[node]['degree'] = block_graph.degree[node]
        # print("degree", block_graph.nodes[node]['degree'])

        block_graph.nodes[node]['concavity'] = area / convex_hull_area if convex_hull_area > 0 else 0
        # print("concavity", block_graph.nodes[node]['concavity'])
        # print("convex_hull_area", convex_hull_area, convex_hull)

        block_graph.nodes[node]['rectanglarity'] = area / min_rot_rec_area if min_rot_rec_area > 0 else 0
        # print("rectanglarity", block_graph.nodes[node]['rectanglarity'])
        # print("min_rot_rec_area", min_rot_rec_area)
        # print()
        # print()



    for edge in block_graph.edges:
        node_a = edge[0]
        node_b = edge[1]
        block_a = block_graph.nodes[node_a]['geometry']
        block_b = block_graph.nodes[node_b]['geometry']
        centroid_a = block_a.centroid
        centroid_b = block_b.centroid

        dx = centroid_b.x - centroid_a.x
        dy = centroid_b.y - centroid_a.y

        block_a_translated = translate(block_a, xoff=dx, yoff=dy)
        area_a = block_a_translated.area
        area_b = block_b.area

        symmetric_diff = block_a_translated.symmetric_difference(block_b)
        symmetric_diff_area = symmetric_diff.area
        r_sim = 1 - (symmetric_diff_area / (area_a + area_b))
        r_sim = max(0.0, min(1.0, r_sim))
        block_graph.edges[edge]['shape_similarity'] = r_sim

        def main_dir(polygon):
            if polygon.geom_type == 'MultiPolygon':
              coords = list(list(polygon.geoms)[0].exterior.coords)
            else:
              coords = list(polygon.exterior.coords)

            if np.allclose(coords[0], coords[-1]):
                coords = coords[:-1]

            centroid = np.mean(coords, axis=0)
            centered_coords = coords - centroid
            cov_matrix = np.cov(centered_coords, rowvar=False)
            eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
            main_eigenvector = eigenvectors[:, np.argmax(eigenvalues)]

            angle_rad = np.arctan2(main_eigenvector[1], main_eigenvector[0])
            angle_deg = np.degrees(angle_rad) % 180
            return angle_deg

        def acute_angle(angle1, angle2):
            diff = abs(angle1 - angle2) % 180
            return min(diff, 180 - diff)

        para_a = main_dir(block_a)
        para_b = main_dir(block_b)
        arrangement_degree = acute_angle(para_a, para_b)
        block_graph.edges[edge]['arrangement_degree'] = min(arrangement_degree, 90 - arrangement_degree)

    return block_graph

# dataset

In [ ]:
# random.seed(42)

In [ ]:
class BlockDataset(Dataset):

    def __init__(self, block_graphs_dict):
        self.blocks = []
        self.cell_ids = []

        for cell_id, block_data in block_graphs_dict.items():
            self.blocks.append(block_data)
            self.cell_ids.append(cell_id)

    def __len__(self):
        return len(self.blocks)

    def __getitem__(self, idx):
        block_data = self.blocks[idx]
        G = block_data['graph']
        polygon = block_data['polygon']

        G_with_features = self._calculate_features(G, polygon)

        gnn0_data = self._graph_to_pyg_data(G_with_features)

        data = {
            'gnn0': gnn0_data,
            # 'label': torch.tensor(0, dtype=torch.long)
            # 'label': torch.tensor(idx % 6, dtype=torch.long)
            'label': torch.tensor(random.randint(0, 6), dtype=torch.long)
        }

        return data

    def _calculate_features(self, G, polygon):
        dict_blocks = {'graph': G, 'polygon': polygon}
        gdf_blocks = extract_street_blocks(dict_blocks)
        block_graph, gdf_blocks = create_street_block_graph(gdf_blocks, G)
        block_graph_with_features = compute_features(block_graph, gdf_blocks, G)
        return block_graph_with_features

    def _graph_to_pyg_data(self, G):
        node_mapping = {node: i for i, node in enumerate(G.nodes())}

        node_features = []
        for node in G.nodes():
            features = []
            node_data = G.nodes[node]
            features.extend([
                node_data.get('number_of_linestrings', 0),
                node_data.get('area', 0),
                node_data.get('circuity', 0),
                node_data.get('concavity', 0),
                node_data.get('rectanglarity', 0),
                node_data.get('degree', 0),
                node_data.get('formfacter', 0),
                node_data.get('elongation', 0)
            ])
            node_features.append(features)


        x = torch.tensor(node_features, dtype=torch.float32)

        edge_index = []
        edge_attr = []

        G_undirected = G.to_undirected()
        for u, v in G_undirected.edges():
            edge_index.append([node_mapping[u], node_mapping[v]])
            edge_index.append([node_mapping[v], node_mapping[u]])

            edge_data = G.get_edge_data(u, v)
            if edge_data:
                arrangement = edge_data.get('arrangement_degree', 0.5)
                similarity = edge_data.get('shape_similarity', 0.5)
                edge_attr.extend([[arrangement, similarity], [arrangement, similarity]])
            else:
                edge_attr.extend([[0.5, 0.5], [0.5, 0.5]])


        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float32)
        # edge_attr = torch.zeros((edge_index.shape[1], 2), dtype=torch.float32)

        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
        # return Data(x=x)

# classify

In [ ]:

def simple_collate_fn(batch):
    """Упрощенная collate функция только для gnn0"""
    batched_data = {}

    # GNN0 данные
    graph_data_list = [item['gnn0'] for item in batch]
    batched_data['gnn0'] = Batch.from_data_list(graph_data_list)

    # Метки
    labels = torch.tensor([item['label'] for item in batch], dtype=torch.long)
    batched_data['label'] = labels

    return batched_data

In [ ]:
def classify_blocks(block_graphs_dict, model_path='best_model.pth', device='cuda'):

    dataset = BlockDataset(block_graphs_dict)
    print(f"Подготовлено {len(dataset)} блоков с корректными признаками")

    # DataLoader
    loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=simple_collate_fn)

    # Загрузка модели
    config = {
        'gnn0': possible_models['gnn0'],
        'label': possible_models['label']
    }

    device = torch.device(device)
    model = DynamicModel(config, num_classes=6).to(device)

    try:
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=False))
        print("Модель загружена")
    except:
        state_dict = torch.load(model_path, map_location=device, weights_only=False)
        model.load_state_dict(state_dict, strict=True)
        print("Модель загружена частично")

    model.eval()

    # Классификация
    predictions = {}
    probabilities = {}

    with torch.no_grad():
        for batch_idx, data in tqdm(enumerate(loader), desc="Processing batches", total=len(loader)):
            data['gnn0'].x = data['gnn0'].x.to(device)
            data['gnn0'].edge_index = data['gnn0'].edge_index.to(device)
            data['gnn0'].edge_attr = data['gnn0'].edge_attr.to(device)
            data['gnn0'].batch = data['gnn0'].batch.to(device)

            output = model(data)
            probs = torch.softmax(output, dim=1)
            _, preds = output.max(1)

            batch_size = len(preds)
            for i in range(batch_size):
                global_idx = batch_idx * 32 + i
                if global_idx < len(dataset):
                    cell_id = dataset.cell_ids[global_idx]
                    predictions[cell_id] = preds[i].item()
                    probabilities[cell_id] = probs[i].cpu().numpy()

    return predictions, probabilities


# start

In [ ]:
# place = "Paris, France"
# place = "Barcelona, Catalonia, Spain"
# place = "Chicago, Illinois, USA"
# place = "Saint Petersburg, Russia"
# place = "Oakland, California, USA"
# place = 'Helsinki, Finland'
# place = "Singapore, Singapore"
place = "Salt Lake City, UT"


G = ox.graph_from_place(place, network_type="drive", simplify=True)
G = ox.project_graph(G)

print(f"Исходный граф: {len(G.nodes)} узлов, {len(G.edges)} ребер")

In [ ]:
print(G.graph['crs'])

In [ ]:
subgraphs = split_graph_by_grid(G, grid_step=1000)

In [ ]:
len(subgraphs)

In [ ]:
torch.cuda.is_available()

In [ ]:
predictions_blocks, probabilities_blocks = classify_blocks(
    subgraphs,
    model_path='best_model.pth',
    device='cuda'
)

In [ ]:

class_names = [
    'Regular Grid', 'Irregular Grid', 'Broken Grid',
    'Warped Parallel', 'Loops & Lollipops', 'Sparse'
]

print("\nРезультаты классификации БЛОКОВ:")
for i, (cell_id, pred_class) in enumerate(list(predictions_blocks.items())[:10]):
    prob = probabilities_blocks[cell_id]
    confidence = prob[pred_class]
    print(f"Блок {cell_id}: {class_names[pred_class]} (уверенность: {prob}")

from collections import Counter
class_counts = Counter(predictions_blocks.values())
print(f"\nСтатистика по классам:")
for class_id in range(6):
    count = class_counts.get(class_id, 0)
    percentage = (count / len(predictions_blocks)) * 100 if predictions_blocks else 0
    print(f"{class_names[class_id]}: {count} блоков ({percentage:.1f}%)")

# visual

## func

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def simple_visualize_classification(subgraphs_dict, predictions, class_names):
    """
    Простая визуализация подграфов по классам - одна картинка
    """

    # Группируем подграфы по классам
    class_subgraphs = {class_id: [] for class_id in range(len(class_names))}

    for cell_id, pred_class in predictions.items():
        if cell_id in subgraphs_dict:
            class_subgraphs[pred_class].append((cell_id, subgraphs_dict[cell_id]))

    # Создаем одну большую фигуру
    n_classes = len([c for c in class_subgraphs.values() if c])  # Только непустые классы
    fig = plt.figure(figsize=(20, 4 * n_classes))

    row = 0
    for class_id, subgraphs_list in class_subgraphs.items():
        if not subgraphs_list:
            continue

        # Ограничиваем количество подграфов для визуализации
        n_to_show = min(8, len(subgraphs_list))

        # Добавляем заголовок класса
        plt.figtext(0.02, 0.95 - row * 0.25,
                   f"{class_names[class_id]} ({len(subgraphs_list)} подграфов)",
                   fontsize=16, fontweight='bold',
                   bbox=dict(boxstyle="round,pad=0.3", facecolor='lightblue'))

        # Визуализируем подграфы этого класса
        for idx, (cell_id, G) in enumerate(subgraphs_list[:n_to_show]):
            ax = plt.subplot(n_classes, n_to_show, row * n_to_show + idx + 1)
            plot_simple_subgraph(ax, G, cell_id, class_names[class_id])

        row += 1

    plt.tight_layout()
    plt.show()

def plot_simple_subgraph(ax, G, cell_id, class_name):
    """Простая визуализация одного подграфа"""
    try:
        # Получаем координаты
        all_x = [G.nodes[node]['x'] for node in G.nodes()]
        all_y = [G.nodes[node]['y'] for node in G.nodes()]

        if not all_x or not all_y:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            return

        # Границы с отступом
        margin = 0.0001
        x_min, x_max = min(all_x) - margin, max(all_x) + margin
        y_min, y_max = min(all_y) - margin, max(all_y) + margin

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

        # Рисуем дороги
        for u, v in G.edges():
            if u in G.nodes() and v in G.nodes():
                u_x, u_y = G.nodes[u]['x'], G.nodes[u]['y']
                v_x, v_y = G.nodes[v]['x'], G.nodes[v]['y']
                ax.plot([u_x, v_x], [u_y, v_y], 'k-', linewidth=2, alpha=0.8)

        # Рисуем узлы
        for node in G.nodes():
            x, y = G.nodes[node]['x'], G.nodes[node]['y']
            ax.plot(x, y, 'ro', markersize=4, alpha=0.7)

        # Заголовок
        ax.set_title(f'{cell_id}', fontsize=8, pad=2)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal')

    except Exception as e:
        ax.text(0.5, 0.5, f'Error', ha='center', va='center', transform=ax.transAxes, fontsize=6)

def visualize_all_subgraphs_simple(subgraphs_dict, predictions, class_names):
    """
    Визуализация всех подграфов на одной картинке с цветовой кодировкой классов
    """
    # Цвета для классов
    colors = plt.cm.Paired(np.linspace(0, 1, len(class_names)))

    fig, ax = plt.subplots(1, 1, figsize=(20, 30))

    # Визуализируем все подграфы
    for cell_id, graph_dict in subgraphs_dict.items():
        G = graph_dict['graph']
        if cell_id in predictions:
            pred_class = predictions[cell_id]
            color = colors[pred_class]

            # Визуализируем подграф
            plot_subgraph_with_color(ax, G, color, class_names[pred_class])

    # Легенда
    legend_elements = []
    for class_id, class_name in enumerate(class_names):
        legend_elements.append(
            plt.Line2D([0], [0], marker='o', color='w',
                      markerfacecolor=colors[class_id], markersize=10, label=class_name)
        )

    ax.legend(handles=legend_elements, loc='upper right')
    ax.set_title('Все подграфы с цветовой кодировкой классов', fontsize=16, fontweight='bold')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

def plot_subgraph_with_color(ax, G, color, class_name):
    """Визуализация подграфа с заданным цветом"""
    try:
        # Рисуем дороги
        for u, v in G.edges():
            if u in G.nodes() and v in G.nodes():
                u_x, u_y = G.nodes[u]['x'], G.nodes[u]['y']
                v_x, v_y = G.nodes[v]['x'], G.nodes[v]['y']
                ax.plot([u_x, v_x], [u_y, v_y], color=color, linewidth=1, alpha=0.6)

        # Рисуем узлы
        for node in G.nodes():
            x, y = G.nodes[node]['x'], G.nodes[node]['y']
            ax.plot(x, y, 'o', color=color, markersize=2, alpha=0.8)

    except:
        pass

def show_class_examples(subgraphs_dict, predictions, class_names):
    """
    Показывает по 2 примера каждого класса
    """
    # Группируем подграфы по классам
    class_subgraphs = {class_id: [] for class_id in range(len(class_names))}

    for cell_id, pred_class in predictions.items():
        if cell_id in subgraphs_dict:
            class_subgraphs[pred_class].append((cell_id, subgraphs_dict[cell_id]))

    # Создаем фигуру
    n_classes = len([c for c in class_subgraphs.values() if c])
    fig, axes = plt.subplots(n_classes, 2, figsize=(10, 3 * n_classes))

    if n_classes == 1:
        axes = axes.reshape(1, -1)

    row = 0
    for class_id, subgraphs_list in class_subgraphs.items():
        if not subgraphs_list:
            continue

        # Берем первые 2 примера
        examples = subgraphs_list[:2]

        for col, (cell_id, G) in enumerate(examples):
            ax = axes[row, col] if n_classes > 1 else axes[col]
            plot_detailed_example(ax, G, cell_id, class_names[class_id])

        row += 1

    # Убираем пустые subplots
    for i in range(row, axes.shape[0] if n_classes > 1 else 1):
        for j in range(2):
            if n_classes > 1:
                axes[i, j].axis('off')
            else:
                if j >= len(examples):
                    axes[j].axis('off')

    plt.tight_layout()
    plt.show()

def plot_detailed_example(ax, G, cell_id, class_name):
    """Детальная визуализация примера"""
    try:
        # Получаем координаты
        all_x = [G.nodes[node]['x'] for node in G.nodes()]
        all_y = [G.nodes[node]['y'] for node in G.nodes()]

        if not all_x or not all_y:
            return

        # Границы
        margin = 0.0001
        x_min, x_max = min(all_x) - margin, max(all_x) + margin
        y_min, y_max = min(all_y) - margin, max(all_y) + margin

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

        # Рисуем дороги
        for u, v in G.edges():
            if u in G.nodes() and v in G.nodes():
                u_x, u_y = G.nodes[u]['x'], G.nodes[u]['y']
                v_x, v_y = G.nodes[v]['x'], G.nodes[v]['y']
                ax.plot([u_x, v_x], [u_y, v_y], 'navy', linewidth=3, alpha=0.8)

        # Рисуем узлы
        for node in G.nodes():
            x, y = G.nodes[node]['x'], G.nodes[node]['y']
            degree = G.degree(node)
            size = 20 + min(degree, 5) * 5
            ax.plot(x, y, 'ro', markersize=size/10, alpha=0.8)

        # Заголовок
        ax.set_title(f'{class_name}\n{cell_id}', fontsize=10, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal')

        # Сетка
        ax.grid(True, alpha=0.3)

    except Exception as e:
        ax.text(0.5, 0.5, f'Error', ha='center', va='center', transform=ax.transAxes)


## start

In [ ]:
class_names = [
    'Regular Grid', 'Irregular Grid', 'Broken Grid',
    'Warped Parallel', 'Loops & Lollipops', 'Sparse'
]

# simple_visualize_classification(subgraphs, predictions_blocks, class_names)

visualize_all_subgraphs_simple(subgraphs, predictions_blocks, class_names)

# show_class_examples(subgraphs, predictions_blocks, class_names)

# # Простая статистика
# from collections import Counter
# class_counts = Counter(predictions_blocks.values())
# print(f"\nСтатистика классификации:")
# for class_id, class_name in enumerate(class_names):
#     count = class_counts.get(class_id, 0)
#     print(f"  {class_name}: {count} подграфов")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_features(block_graphs_dict):
    """Анализ распределения признаков с автоматическим масштабированием"""
    all_features = []

    for cell_id, block_data in block_graphs_dict.items():
        G = block_data['graph']
        polygon = block_data['polygon']

        dict_blocks = {'graph': G, 'polygon': polygon}
        gdf_blocks = extract_street_blocks(dict_blocks)
        block_graph, gdf_blocks = create_street_block_graph(gdf_blocks, G)
        block_graph_with_features = compute_features(block_graph, gdf_blocks, G)

        for node in block_graph_with_features.nodes():
            features = block_graph_with_features.nodes[node]
            all_features.append({
                'cell_id': cell_id,
                'node_id': node,
                'area': features.get('area', 0),
                'circuity': features.get('circuity', 0),
                'concavity': features.get('concavity', 0),
                'number_of_linestrings': features.get('number_of_linestrings', 0),
                'formfacter': features.get('formfacter', 0),
                'rectanglarity': features.get('rectanglarity', 0),
                'elongation': features.get('elongation', 0),
                'degree': features.get('degree', 0),
            })

    df = pd.DataFrame(all_features)

    # Визуализация распределения признаков
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.flatten()

    feature_columns = ['area', 'circuity', 'concavity', 'number_of_linestrings',
                       'formfacter', 'rectanglarity', 'elongation', 'degree']

    for i, feature in enumerate(feature_columns):
        if i < len(axes):
            data = df[feature].dropna()

            # Автоматически определяем разумные границы
            # Используем 5-й и 95-й процентили для удаления выбросов
            if len(data) > 0:
                q05 = data.quantile(0.05)
                q95 = data.quantile(0.95)

                # Фильтруем данные для визуализации
                filtered_data = data[(data >= q05) & (data <= q95)]

                # Если после фильтрации что-то осталось
                if len(filtered_data) > 0:
                    axes[i].hist(filtered_data, bins=50, edgecolor='black', alpha=0.7)
                    axes[i].set_title(feature)
                    axes[i].set_xlabel('Value')
                    axes[i].set_ylabel('Frequency')

                    # Автоматически устанавливаем границы
                    x_min = filtered_data.min()
                    x_max = filtered_data.max()

                    # Добавляем небольшой отступ по краям (10%)
                    margin = (x_max - x_min) * 0.1
                    axes[i].set_xlim(x_min - margin, x_max + margin)

                    # Форматирование чисел на оси X
                    if x_max > 1000:  # Для больших чисел используем научную нотацию
                        axes[i].ticklabel_format(axis='x', style='sci', scilimits=(-3, 3))
                else:
                    axes[i].text(0.5, 0.5, 'No data',
                               horizontalalignment='center',
                               verticalalignment='center',
                               transform=axes[i].transAxes)
                    axes[i].set_title(feature)
            else:
                axes[i].text(0.5, 0.5, 'No data',
                           horizontalalignment='center',
                           verticalalignment='center',
                           transform=axes[i].transAxes)
                axes[i].set_title(feature)

    plt.tight_layout()
    plt.show()

    return df

# Вызовите эту функцию перед классификацией
df_features = analyze_features(subgraphs)

In [ ]:
import pandas as pd
import seaborn as sns

def analyze_features(block_graphs_dict):
    """Анализ распределения признаков"""
    all_features = []

    for cell_id, block_data in block_graphs_dict.items():
        G = block_data['graph']
        polygon = block_data['polygon']

        dict_blocks = {'graph': G, 'polygon': polygon}
        gdf_blocks = extract_street_blocks(dict_blocks)
        block_graph, gdf_blocks = create_street_block_graph(gdf_blocks, G)
        block_graph_with_features = compute_features(block_graph, gdf_blocks, G)

        for node in block_graph_with_features.nodes():
            features = block_graph_with_features.nodes[node]
            all_features.append({
                'cell_id': cell_id,
                'node_id': node,
                'area': features.get('area', 0),
                'circuity': features.get('circuity', 0),
                'concavity': features.get('concavity', 0),
                'number_of_linestrings': features.get('number_of_linestrings', 0),
                'formfacter': features.get('formfacter', 0),
                'rectanglarity': features.get('rectanglarity', 0),
                'elongation': features.get('elongation', 0),
                'degree': features.get('degree', 0),
            })

    df = pd.DataFrame(all_features)

    # Визуализация распределения признаков
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.flatten()

    feature_columns = ['area', 'circuity', 'concavity', 'number_of_linestrings',
                       'formfacter', 'rectanglarity', 'elongation', 'degree']

    for i, feature in enumerate(feature_columns):
        if i < len(axes):
            axes[i].hist(df[feature].dropna(), bins=100, edgecolor='black', alpha=0.7)
            axes[i].set_title(feature)
            axes[i].set_xlabel('Value')
            axes[i].set_ylabel('Frequency')

    plt.tight_layout()
    plt.show()

    return df

# Вызовите эту функцию перед классификацией
df_features = analyze_features(subgraphs)